# Session 2 - black box models

How do we inspect a stronger black-box model without confusing explanation with causation?

**Learning goals**
- Train and evaluate a black-box classifier on Adult income data
- Create global explanations with permutation importance
- Create feature-effect explanations with PDP/ICE


We plan to use the Adult census data here again because it offers comparative insight between session 1 and 2, especially since we are exploring explainability, the idea is to trace across the methods explored where does explainability comfortability sit in terms of outputs presented so far in the learning based demo codebooks.

**Methods used in this notebook**
- Random Forest
- Permutation Importance
- PDP / ICE

In [2]:
from pathlib import Path
import warnings
import importlib.util
import random
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.inspection import PartialDependenceDisplay
from sklearn.inspection import partial_dependence

SEED = 42
np.random.seed(SEED)


In [ ]:
CODE_DIR = Path.cwd()
REPO_ROOT = CODE_DIR.parent

DATA_DIR = REPO_ROOT / "Data"
OUTPUT_DIR = REPO_ROOT / "Outputs" / "session_2"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
EXPLAIN_DIR = OUTPUT_DIR / "explanations"

for folder in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR, EXPLAIN_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Added sub-folder file structure for outputs from session 1

## Dataset recap

Adult census data (Tabular)

| Prediction task is to determine whether a person makes over 50K a year from 14 available features, numerical and categorical.

The prediction task is to **classify** based in the features into two classes
-  `income` binary target indicating whether annual income is `>50K` or `<=50K`

UCIrvine: [Adult census data](https://archive.ics.uci.edu/dataset/2/adult)
- Becker, B. & Kohavi, R. (1996). Adult [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20.

In [ ]:
from sklearn.model_selection import train_test_split

#loading up and check
adult_path = DATA_DIR / "adult.data"

df = pd.read_csv(adult_path)

print("Shape:", df.shape)
display(df.head())

print("\nColumns:")
print(df.columns.tolist())

## we will not perform many inspections in this codebook since it is familiar from session 1.

Adding header items, so we can refer to them easily.

In [ ]:
# These are the official Adult dataset variable names, added manually because
# adult.data does not include a header row.
adult_columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

# Path to the raw UCI Adult data file inside the repo Data folder
adult_data_path = DATA_DIR / "adult.data"

# Read the raw file into a pandas DataFrame
df_adult_raw = pd.read_csv(
    adult_data_path,
    header=None,             # there is no header row in adult.data
    names=adult_columns,     # so we assign the column names ourselves
    na_values="?",           # in this dataset, ? is used for some missing values
    skipinitialspace=True    # removes spaces after commas
)

print("Shape:", df_adult_raw.shape)
display(df_adult_raw.head())

print("\nMissing values per column:")
display(df_adult_raw.isna().sum().sort_values(ascending=False))

# Modelling data frame

In [ ]:
# Create a modelling copy so the raw imported dataset remains unchanged
df_adult_model = df_adult_raw.copy()

# Clean the income labels just in case there are extra spaces or punctuation
df_adult_model["income"] = (
    df_adult_model["income"]
    .astype(str)
    .str.strip()
    .str.replace(".", "", regex=False)
)

# Map the target column to binary values for classification
# <=50K becomes 0 and >50K becomes 1
df_adult_model["income"] = df_adult_model["income"].map({
    "<=50K": 0,
    ">50K": 1
})

print("Model dataframe shape:", df_adult_model.shape)
print("\nIncome value counts:")
print(df_adult_model["income"].value_counts(dropna=False))

# Define the full feature matrix and target vector
X_adult_full = df_adult_model.drop(columns="income")
y_adult_full = df_adult_model["income"]

print("\nFull feature matrix shape:", X_adult_full.shape)
print("Full target vector shape:", y_adult_full.shape)

# Create train and test splits
X_adult_train, X_adult_test, y_adult_train, y_adult_test = train_test_split(
    X_adult_full,
    y_adult_full,
    test_size=0.2,
    random_state=SEED,
    stratify=y_adult_full
)

print("\nTrain/test split shapes:")
print("X_adult_train:", X_adult_train.shape)
print("X_adult_test :", X_adult_test.shape)
print("y_adult_train:", y_adult_train.shape)
print("y_adult_test :", y_adult_test.shape)

In [8]:
# Save the cleaned modelling dataframe for reuse later in Session 2
session2_01_cleaned_adult_data_path = TABLE_DIR / "session2_01_cleaned_adult_data.csv"
df_adult_model.to_csv(session2_01_cleaned_adult_data_path, index=False)

In [9]:
# Save train/test splits for traceability and reuse later in Session 2
session2_02_X_adult_train_path = TABLE_DIR / "session2_02_X_adult_train.csv"
session2_03_X_adult_test_path = TABLE_DIR / "session2_03_X_adult_test.csv"
session2_04_y_adult_train_path = TABLE_DIR / "session2_04_y_adult_train.csv"
session2_05_y_adult_test_path = TABLE_DIR / "session2_05_y_adult_test.csv"

X_adult_train.to_csv(session2_02_X_adult_train_path, index=False)
X_adult_test.to_csv(session2_03_X_adult_test_path, index=False)
y_adult_train.to_csv(session2_04_y_adult_train_path, index=False)
y_adult_test.to_csv(session2_05_y_adult_test_path, index=False)

## Build the preprocessing and modelling pipeline for Random Forest

Both categorical and numerical values exist in the dataset. 

Combine preprocessing and modelling in one scikit-learn pipeline.
This keeps the workflow tidy and avoids accidental leakage between training and test data.

In [ ]:
# Separate numeric and categorical columns from the Adult training set
adult_numeric_features = X_adult_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
adult_categorical_features = X_adult_train.select_dtypes(include=["object", "string"]).columns.tolist()

print("Numeric features:")
print(adult_numeric_features)

print("\nCategorical features:")
print(adult_categorical_features)

# Numeric preprocessing:
# fill missing numeric values with the median value from the training data
adult_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing:
# fill missing values with the label "missing"
# then one-hot encode categories into numeric columns
adult_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine numeric and categorical preprocessing into one reusable preprocessing object
adult_preprocessor_base = ColumnTransformer(
    transformers=[
        ("num", adult_numeric_transformer, adult_numeric_features),
        ("cat", adult_categorical_transformer, adult_categorical_features)
    ]
)

# Random Forest model
rf_model_adult = RandomForestClassifier(
    n_estimators=200,   # how many trees we want in the forest
    random_state=SEED,  # use the seed defined earlier so results are reproducible
    n_jobs=-1           
)

# Full preprocessing + model pipeline
rf_pipeline_adult = Pipeline(steps=[
    ("preprocessor", adult_preprocessor_base),
    ("model", rf_model_adult)
])

rf_pipeline_adult

# Fit Random Forest

## Random Forest as an opaque system

A Random Forest builds many decision trees and combines their predictions. We saw a simple (shallow tree) last session

If we write the prediction from tree $b$ as $T_b(x)$, then the forest prediction can be written as:


$$
\hat{p}(y=1 \mid x) = \frac{1}{B} \sum_{b=1}^{B} T_b(x)
$$

Breakdown

- $x$ is one observation
- $B$ is the number of trees in the forest
- $T_b(x)$ is the prediction from tree $b$

For classification, the final class prediction is based on the combined votes or averaged class probabilities across the trees:

$$
\hat{y}(x) = \arg\max_{c} \; \frac{1}{B} \sum_{b=1}^{B} \mathbf{1}\{T_b(x)=c\}
$$


In [ ]:
# Fit the full Random Forest pipeline on the Adult training data
#TRAINING ON THE TRAINING SET
rf_pipeline_adult.fit(X_adult_train, y_adult_train)

#TEST SET
# Generate predictions on both training and test sets
y_adult_train_pred_rf = rf_pipeline_adult.predict(X_adult_train)
y_adult_test_pred_rf = rf_pipeline_adult.predict(X_adult_test)

# Predicted probabilities for the positive class
y_adult_train_proba_rf = rf_pipeline_adult.predict_proba(X_adult_train)[:, 1]
y_adult_test_proba_rf = rf_pipeline_adult.predict_proba(X_adult_test)[:, 1]

# Build a compact metrics table
rf_metrics_adult = pd.DataFrame({
    "dataset": ["train", "test"],
    "accuracy": [
        accuracy_score(y_adult_train, y_adult_train_pred_rf),
        accuracy_score(y_adult_test, y_adult_test_pred_rf)
    ],
    "roc_auc": [
        roc_auc_score(y_adult_train, y_adult_train_proba_rf),
        roc_auc_score(y_adult_test, y_adult_test_proba_rf)
    ]
})

print("Random Forest metrics:")
display(rf_metrics_adult)

print("\nTest classification report:")
print(classification_report(y_adult_test, y_adult_test_pred_rf))

rf_confusion_matrix_adult = confusion_matrix(y_adult_test, y_adult_test_pred_rf)
print("\nTest(set) confusion matrix:")
print(rf_confusion_matrix_adult)

In [ ]:
# Save the fitted Random Forest pipeline for reuse in Session 2B
session2_20_rf_pipeline_adult_path = OUTPUT_DIR / "session2_20_rf_pipeline_adult.joblib"
joblib.dump(rf_pipeline_adult, session2_20_rf_pipeline_adult_path)

In [ ]:
# Plot the test confusion matrix for the Random Forest Adult model
plt.figure(figsize=(6, 5))

rf_confusion_matrix_display_adult = ConfusionMatrixDisplay.from_predictions(
    y_adult_test,
    y_adult_test_pred_rf,
    display_labels=["<=50K", ">50K"],
    cmap="Blues",
    values_format="d"
)

plt.title("Random Forest Test Confusion Matrix (Adult)")
plt.tight_layout()
plt.show()

# Save the test confusion matrix plot for Session 2
session2_09_rf_adult_test_confusion_matrix_plot_path = (
    FIG_DIR / "session2_06_rf_adult_test_confusion_matrix_plot.png"
)

rf_confusion_matrix_display_adult.figure_.savefig(
    session2_09_rf_adult_test_confusion_matrix_plot_path,
    dpi=300,
    bbox_inches="tight"
)

- **Rows = the true class**
- **Columns = the model's predicted class**

## Train and Test splits

We started with **32,561 rows** in the modelling dataset (the output from # Modelling data frame) and **14 predictor variables** plus **1 target column** (`income`).


We then split the data into:

- **Training set:** 26,048 rows (**80%**)
- **Test set:** 6,513 rows (**20%**)

Used **stratified splitting**

### Class balance after the split

**Training set**
- Income `0` (<=50K): **19,775**
- Income `1` (>50K): **6,273**

**Test set**
- Income `0` (<=50K): **4,945**
- Income `1` (>50K): **1,568**

The test set should look like the original problem distribution while still remaining **unseen during training**.

In this notebook, we set:

$$
B = 200
$$


Saving things for reference

In [17]:
# Save the test set confusion matrix for the Random Forest Adult model
rf_confusion_matrix_adult_df = pd.DataFrame(
    rf_confusion_matrix_adult,
    index=["true_0", "true_1"],
    columns=["pred_0", "pred_1"]
)

session2_08_rf_adult_test_confusion_matrix_path = (
    TABLE_DIR / "session2_08_rf_adult_test_confusion_matrix.csv"
)

rf_confusion_matrix_adult_df.to_csv(
    session2_08_rf_adult_test_confusion_matrix_path,
    index=True
)

## Global model inspection with Permutation Feature Importance

the Random Forest has been trained and evaluated on a held-out test set, let's inspect which features the model relies on most. Permutation Feature Importance (PFI) is a **global** explanation method. The purpose is to keep the fitted model fixed- shuffle one feature at a time in the evaluation data- measure how much model performance drops If shuffling a feature causes a large drop in performance, that feature was important for the model.If shuffling a feature causes little or no change, that feature mattered less to the fitted model.

PFI helps with:**Which input features does this trained model depend on most overall?**

This is different from local explanation methods such as SHAP or LIME (we will meet them later), which explain **individual predictions** rather than the model's overall reliance on features.

> Remember: Features are the items we saw in the beginning, capital gains, age, country of origin and such. 

### Important interpretation note

Permutation importance tells us about **model reliance**. A feature can appear important because the model uses it, even if the feature is only correlated with the target (which was our income binary) rather than causally responsible for it. It is also worth remembering that if features are strongly correlated, permutation importance can be harder to interpret because shuffling one feature may not fully remove the predictive information available through related features.


In [ ]:
# Compute permutation feature importance on the held-out test set
rf_permutation_importance_adult = permutation_importance(
    rf_pipeline_adult,
    X_adult_test,
    y_adult_test,
    n_repeats=20,          # repeat shuffling to get a more stable estimate
    random_state=SEED,     # reproducible permutation results
    scoring="roc_auc",     # match the evaluation metric used earlier - note this for later when we review the feature importance
    n_jobs=-1              
)

# Store the permutation importance results in a tidy dataframe
rf_permutation_importance_adult_df = pd.DataFrame({
    "feature": X_adult_test.columns,
    "importance_mean": rf_permutation_importance_adult.importances_mean,
    "importance_std": rf_permutation_importance_adult.importances_std
}).sort_values("importance_mean", ascending=False)

print("PFI(top 15 features):")
display(rf_permutation_importance_adult_df.head(15))

## Review before moving from model evaluation to inspection

Before interpreting the PFI output, it helps to pause and review what has already happened.

At this stage, we already have a **trained Random Forest pipeline**. This means:

- the preprocessing steps were learned from the training data
- the Random Forest was then fitted on those processed training inputs
- we are now in the **inspection stage**, using the held-out test set rather than retraining the model

### What does Permutation Feature Importance do?

It speaks to:

**How much worse does the fitted model perform if we break one feature at a time?**

To answer that, the following sequence is used:

1. Start with the **held-out test set** in its original form.
2. Measure the model's **baseline performance** on that untouched test set.
3. Pick one feature column and **shuffle its values**.
4. Pass that shuffled test set through the **already fitted pipeline** again.
5. Inside the pipeline, the preprocessing is applied again, including one-hot encoding for categorical variables.
6. The Random Forest then produces prediction scores from those transformed inputs.
7. The model is scored again using **ROC AUC**.
8. The drop in score is recorded as the importance of that feature.
9. This process is repeated several times for the same feature, and then repeated for each feature in turn.

### What was the baseline?

Because we set:

python
scoring="roc_auc"

### What about  `n_repeats=20`?

If we shuffled a feature only once, the result could be noisy.  
So we shuffle each feature **20 times**, measure the score drop each time, and then report:

- importance_mean = the average drop in ROC AUC
- importance_std = how much that drop varied across the repeats

This gives a more stable estimate of importance. scikit-learn returns both the mean and standard deviation of the importances across repeated permutations.

### Important interpretation note

Even though the Random Forest was fitted on the **one-hot encoded feature space inside the pipeline**, the PFI table shown here is still at the **original input feature level**.

That is because we passed the **full pipeline** and the **original test dataframe** into permutation_importance. So each importance value answers:

**What happens if we shuffle this original input column, then let the whole fitted pipeline process it?**

This makes the result easier to interpret, because it stays at the same level as the original dataset columns. The pipeline transforms the data before prediction, but the permutation step acts on the columns of the X that we provided.

You do not need to run all saves, its for further learning exploration and storing outputs

In [23]:
# Save the permutation importance table 
session2_09_rf_adult_permutation_importance_path = (
    TABLE_DIR / "session2_09_rf_adult_permutation_importance.csv"
)

rf_permutation_importance_adult_df.to_csv(
    session2_09_rf_adult_permutation_importance_path,
    index=False
)

In [ ]:
# Plot the top permutation importance results
rf_permutation_importance_top15_adult = (
    rf_permutation_importance_adult_df.head(15)
    .sort_values("importance_mean", ascending=True)
)

plt.figure(figsize=(8, 6))
plt.barh(
    rf_permutation_importance_top15_adult["feature"],
    rf_permutation_importance_top15_adult["importance_mean"],
    xerr=rf_permutation_importance_top15_adult["importance_std"]
)
plt.xlabel("Mean decrease in ROC AUC after permutation")
plt.ylabel("Feature")
plt.title("Random Forest Permutation Feature Importance (Adult)")
plt.tight_layout()
plt.show()

# Save the permutation importance figure
session2_10_rf_adult_permutation_importance_plot_path = (
    FIG_DIR / "session2_10_rf_adult_permutation_importance_plot.png"
)


In this plot, the x-axis shows the **mean decrease in ROC AUC after permutation**.

- a **larger positive value** means the model relied more on that feature
- a value **close to zero** means the feature mattered little to the fitted model
- a **negative value** means the score improved slightly after shuffling, which usually suggests noise or redundancy rather than useful signal

Because we used `scoring="roc_auc"`, the importance values are measured in **AUC units**.

For example, if the baseline test ROC AUC is $0.908$ and a feature has permutation importance $0.042$, then after shuffling that feature the model's ROC AUC drops to roughly:

$$
0.908 - 0.042 = 0.866
$$

So the chart tells us how much predictive discrimination the model loses when each feature is disrupted.

The error bars show the variation across repeated shuffles.

### How does Permutation Feature Importance (PFI) work?

It tells us the following:
**How much worse does the model perform if we shuffle one feature and break its relationship with the outcome?**

Mathematically, for feature $j$, we compare the model's original score with the score after permuting that feature:

$$
I_j = s(f, X, y) - \frac{1}{K}\sum_{k=1}^{K} s\left(f, \tilde{X}^{(k)}_j, y\right)
$$

where:

- $I_j$ is the permutation importance of feature $j$
- $s(f, X, y)$ is the model score on the unshuffled data
- $\tilde{X}^{(k)}_j$ is the dataset after feature $j$ has been shuffled in repeat $k$
- $K$ is the number of permutation repeats

### What do we know so far?

So far, the strongest model-reliance signals are:

- $capital\_gain$
- $marital\_status$
- $age$
- $occupation$
- $hours\_per\_week$
- $relationship$

What kind of mixture of features is the Random Forest relying on?

A small negative value for $education$ means that, in this fitted model, shuffling that feature did not hurt performance and may even have improved it slightly. In practice, that usually means the feature adds little unique signal once the other predictors are already available.

In [ ]:
# Baseline ROC AUC on the untouched test set
rf_baseline_test_auc_adult = roc_auc_score(
    y_adult_test,
    rf_pipeline_adult.predict_proba(X_adult_test)[:, 1]
)

print("Baseline test ROC AUC:", round(rf_baseline_test_auc_adult, 6))

# Build a more interpretable performance-breakdown table
rf_permutation_breakdown_adult_df = rf_permutation_importance_adult_df.copy()

rf_permutation_breakdown_adult_df["baseline_test_auc"] = rf_baseline_test_auc_adult
rf_permutation_breakdown_adult_df["mean_auc_after_shuffle"] = (
    rf_baseline_test_auc_adult - rf_permutation_breakdown_adult_df["importance_mean"]
)
rf_permutation_breakdown_adult_df["std_auc_after_shuffle"] = (
    rf_permutation_breakdown_adult_df["importance_std"]
)
rf_permutation_breakdown_adult_df["percent_drop_from_baseline"] = (
    rf_permutation_breakdown_adult_df["importance_mean"] / rf_baseline_test_auc_adult * 100
)

# Reorder columns for readability
rf_permutation_breakdown_adult_df = rf_permutation_breakdown_adult_df[
    [
        "feature",
        "baseline_test_auc",
        "mean_auc_after_shuffle",
        "std_auc_after_shuffle",
        "importance_mean",
        "importance_std",
        "percent_drop_from_baseline",
    ]
]

print("PFI performance breakdown (top 15 features):")
display(rf_permutation_breakdown_adult_df.head(15))

In [ ]:
# Choose one feature to inspect in detail
feature_to_inspect = "capital_gain"

feature_idx = list(X_adult_test.columns).index(feature_to_inspect)

rf_permutation_repeats_adult_df = pd.DataFrame({
    "repeat": range(1, len(rf_permutation_importance_adult.importances[feature_idx]) + 1),
    "auc_drop": rf_permutation_importance_adult.importances[feature_idx],
})

rf_permutation_repeats_adult_df["baseline_test_auc"] = rf_baseline_test_auc_adult
rf_permutation_repeats_adult_df["auc_after_shuffle"] = (
    rf_baseline_test_auc_adult - rf_permutation_repeats_adult_df["auc_drop"]
)
rf_permutation_repeats_adult_df["feature"] = feature_to_inspect

display(rf_permutation_repeats_adult_df)

ROC = Receiver Operating Characteristic curve.
It is a curve showing model performance across many possible classification thresholds by comparing the true positive rate and false positive rate.

AUC = Area Under the Curve.
It is the area under that ROC curve, summarized as one number. scikit-learn defines roc_auc_score as the area under the ROC curve computed from prediction scores.

# Partial Dependence Plots (PDP) and Individual Conditional Expectation (ICE) plots

PDP shows the average effect, while the ICE lines show how individual cases vary around that average.

**How does the model's prediction change when one feature changes?**

### Partial Dependence Plot (PDP)

A **Partial Dependence Plot** shows the **average model prediction** as one feature varies, while the other features are averaged over the dataset.

Mathematically, for a feature (or set of features) $X_S$, the partial dependence function is:

$$
\hat{f}_S(x_S) = \mathbb{E}_{X_C}\left[\hat{f}(x_S, X_C)\right]
$$

- $X_S$ = the feature we want to study
- $X_C$ = all the other features
- $\hat{f}$ = the fitted model

In practice, this expectation is approximated by averaging over the observed dataset:

$$
\hat{f}_S(x_S) \approx \frac{1}{n}\sum_{i=1}^{n}\hat{f}(x_S, x_C^{(i)})
$$

So the PDP line is:

**the average prediction we would get if we set the chosen feature to a specific value for everyone, while keeping all other features as they were.**

### Individual Conditional Expectation (ICE)

An **ICE plot** does something similar, but for **one observation at a time**.

For observation $i$, the ICE curve is:

$$
\hat{f}^{(i)}_S(x_S) = \hat{f}(x_S, x_C^{(i)})
$$

So instead of averaging across all observations, ICE shows:

**how the prediction for one specific case changes as the chosen feature moves across a grid of values**

- **PDP** = one average line
- **ICE** = many individual lines
- the PDP can be thought of as the average of the ICE curves

### What is happening computationally?

For each feature we choose:

1. scikit-learn creates a **grid of values** across that feature
2. it takes the dataset and replaces that feature with one grid value at a time
3. it sends those modified rows through the **already fitted pipeline**
4. the pipeline preprocesses the data again
5. the fitted Random Forest produces prediction scores
6. those scores are either:
   - averaged across rows for the **PDP**
   - kept as separate row-level curves for the **ICE**

In [44]:
# Make a plotting copy so the original test set stays unchanged
X_adult_test_pdp = X_adult_test.copy()

# Convert numeric columns to float for PDP/ICE grid evaluation
X_adult_test_pdp[adult_numeric_features] = X_adult_test_pdp[adult_numeric_features].astype(float)

# Choose important numeric features for PDP/ICE
adult_pdp_features = ["capital_gain", "age", "hours_per_week"] 

## Plot

In [ ]:
for feature_name in adult_pdp_features:
    fig, ax = plt.subplots(figsize=(8, 6))

    PartialDependenceDisplay.from_estimator(
        rf_pipeline_adult,
        X_adult_test_pdp,
        features=[feature_name],
        kind="both",
        subsample=200,
        grid_resolution=30,
        random_state=SEED,
        ax=ax
    )

    ax.set_title(f"Partial Dependence and ICE: {feature_name}")
    plt.tight_layout()
    plt.show()

> Reminder: The partial dependence plot is a global method: The method considers all instances and gives a statement about the global relationship of a feature with the predicted outcome.

The x-axis shows the **feature values** being explored.

- `age` = age values
- `hours_per_week` = hours worked per week
- `capital_gain` = capital gain values

The grid values are generated from the data. In scikit-learn, they are based on the feature values in `X`, using percentiles and a chosen grid resolution. In our code, we set:

- `grid_resolution=30`

so the curve is evaluated at 30 points across the chosen feature range. Deciles of the feature are also shown on the x-axis as small tick marks to indicate where the data are concentrated.

#### y-axis
The y-axis shows the model's **target response**.

For classification, scikit-learn can use either:

- class probability
- or decision function

With `response_method="auto"`, scikit-learn tries `predict_proba` first and uses that if available. Because our fitted pipeline ends in a `RandomForestClassifier`, this means the y-axis is the model's **predicted probability for the positive class**. The y-axis typically runs between 0 and 1.

In our binary income example, the positive class is:
 y=1 which is >50K
**predicted probability that income is greater than 50K**

### How do we interpret the shapes?

- If the **PDP goes upward**, higher values of the feature are associated with higher predicted probability of the positive class.
- If the **PDP goes downward**, higher values of the feature are associated with lower predicted probability.
- If the **PDP is flat**, the model's prediction changes little as that feature changes.
- If the **ICE lines are close together**, the feature has a similar effect across many observations.
- If the **ICE lines spread apart or cross**, the effect of that feature differs across observations, which often suggests interactions with other features.

### Important

PDP and ICE can be misleading when features are strongly correlated.

That is because these methods create artificial combinations of feature values that may be rare or unrealistic in the original data.

So the plots should be interpreted as:

**how the fitted model behaves when we vary one feature in a controlled way**

## Diving into age as an example

In [ ]:
feature_name = "age"

pd_age_raw = partial_dependence(
    rf_pipeline_adult,
    X_adult_test_pdp,
    features=[feature_name],
    kind="both",
    response_method="auto",
    grid_resolution=10
)

age_grid_values = pd_age_raw["grid_values"][0]
age_pdp_values = pd_age_raw["average"][0]
age_ice_values = pd_age_raw["individual"][0]

print("Feature being analysed:", feature_name)
print("Grid values shape:", age_grid_values.shape)
print("PDP values shape:", age_pdp_values.shape)
print("ICE values shape:", age_ice_values.shape)

display(
    pd.DataFrame({
        "age_grid_value": age_grid_values,
        "pdp_average_predicted_probability_gt_50K": age_pdp_values
    })
)

In [ ]:
# Pick one grid value from the PDP calculation
example_age_value = float(age_grid_values[0])

# Make a copy of the test set and force age to this single value for every row
X_age_fixed = X_adult_test_pdp.copy()
X_age_fixed["age"] = example_age_value

# Send the modified data through the fitted pipeline
# The pipeline preprocesses the data and the Random Forest returns probabilities
age_fixed_probabilities = rf_pipeline_adult.predict_proba(X_age_fixed)[:, 1]

print("Chosen age grid value:", example_age_value)
print("First 10 predicted probabilities for income > 50K at this fixed age:")
print(age_fixed_probabilities[:10])

print("\nAverage predicted probability at this grid value:")
print(age_fixed_probabilities.mean())

print("\nCorresponding PDP value from partial_dependence:")
print(age_pdp_values[0])

In [ ]:
# first 5 ICE curves as a dataframe
ice_preview_age_df = pd.DataFrame(
    age_ice_values[:5, :],
    index=[f"sample_{i}" for i in range(5)],
    columns=[f"age_{round(v, 2)}" for v in age_grid_values]
)

display(ice_preview_age_df)

## Interpreting the raw PDP and ICE output for `age`

The raw output for `age` shows three related things:

- `age_grid_value` = the x-axis values used for the plot
- `pdp_average_predicted_probability_gt_50K` = the average predicted probability at each age grid value
- `ICE values` = the individual predicted probabilities for each test-set observation at each age grid value

### 1. What does the PDP table mean?

This table:

- `age_grid_value`
- `pdp_average_predicted_probability_gt_50K`

shows the **average model prediction** when age is fixed to each grid value in turn.

For example:

- at age $19$, the average predicted probability of `income > 50K` is about $0.146$
- at age $53.22$, the average predicted probability is about $0.296$
- by age $63$, it falls slightly to about $0.268$

So the fitted model is learning a pattern in which the predicted probability of higher income:

- rises from younger ages into mid-life
- then levels off
- and then decreases slightly at older ages

This is the **PDP trend**.

### 2. What does the "chosen age grid value" example mean?

When we set:

- `age = 19.0` for **every row** in the test set

and then run the fitted pipeline again, the model gives one predicted probability per row.

That is why we saw values such as:

- `0.145`
- `0.000`
- `0.880`
- `0.570`
- `0.035`

These are **not all the same**, because age was the only thing changed.

All the other features for each person stayed as they originally were, such as:

- education
- occupation
- capital gain
- work hours
- marital status

So even when everyone is forced to age $19$, the model still gives different predictions because the rest of the input profile is different for each row.

### 3. Why does the average equal the PDP value?

After fixing age at $19$ for everyone, we average all of those row-level predicted probabilities:

$$
\frac{1}{n}\sum_{i=1}^{n}\hat{f}(19, x_C^{(i)})
$$

That average was:

$$
0.14587210194994626
$$

and that exactly matches the PDP value for age $19$.

So:

- the **ICE values** are the row-level predictions
- the **PDP value** is the mean of those ICE values at that grid point

### 4. What does the ICE preview table mean?

The ICE table shows the same calculation, but now across many age values.

Each:

- **row** = one observation from the test set
- **column** = one age grid value
- **cell** = the model's predicted probability of `income > 50K` when that observation's age is forced to that value

For example:

- `sample_0` rises from about `0.145` at age `19` to about `0.380` at age `63`
- `sample_1` stays near zero across all ages
- `sample_2` stays very high across all ages
- `sample_3` changes in a less smooth way
- `sample_4` stays low, with only small increases

This tells us that age does **not** affect every observation in the same way.

### 5. What does this  explain?

The PDP shows

**On average, increasing age from younger to middle-age increases the model's predicted probability of higher income.**

The ICE values say:

**That age effect is not identical for every person. Some people are hardly affected by age, while others show much stronger changes.**

So the PDP gives the **average effect**, and the ICE output reveals the **individual variation around that average**.

In [ ]:
# Categorical features, PDP only
adult_pdp_categorical_features = ["education"]

for feature_name in adult_pdp_categorical_features:
    fig, ax = plt.subplots(figsize=(8, 6))

    PartialDependenceDisplay.from_estimator(
        rf_pipeline_adult,
        X_adult_test_pdp,
        features=[feature_name],
        categorical_features=adult_categorical_features,
        kind="average",
        ax=ax
    )

    ax.set_title(f"Partial Dependence (categorical): {feature_name}")
    plt.tight_layout()
    plt.show()

Okay so maybe this data is old because the doctorate has high probability for income in the positive class....

A value like 0.32 means about a 32% predicted chance of income > 50K on average under this forced category.

higher bar means the model associates that category with a higher predicted chance of income > 50K
lower bar means the model associates it with a lower predicted chance of income > 50K.

## Break point and restart for local explanations

We have now completed the **global inspection** part of the workflow:

- model evaluation
- permutation feature importance
- partial dependence plots
- ICE plots

After a short break, we now move to **local explanation methods**.

Local methods focus on **why did the model make this prediction for this specific observation?**

- **SHAP** for local feature attributions
- **LIME** for local surrogate explanations

# Move to codebook B for session 2